In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import mannwhitneyu

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
rnadir = f'{projdir}/snmCT/rna'

In [ ]:
nuclei = pd.read_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t', index_col=0)
early = nuclei[nuclei['early_ipsc']]
nuclei.shape, early.shape

In [ ]:
early_ct = early[early['assay'] == 'snmCT']
early_ct['ips_subtype'].value_counts()

### ED10E: RNA UMAP with retained nuclei only

In [ ]:
adata = sc.read_h5ad(f'{rnadir}/h5ad/processed/rna_finegrain.h5ad')
adata

In [ ]:
umap_df = pd.DataFrame(adata.obsm['X_umap'][:, :2], index=adata.obs.index, columns=['umap0', 'umap1'])
umap_df['ips_subtype'] = early_ct['ips_subtype'].reindex(umap_df.index)
umap_df['ips_subtype'].value_counts()

In [ ]:
umap_df.to_csv(f'{outdir}/snmct_rna_ips_subtype_umap.csv', sep='\t')

### ED10H: cell-cycle scores

In [ ]:
adata.obs['ips_subtype'] = early_ct['ips_subtype'].reindex(adata.obs.index)
ips_cc = adata[~adata.obs['ips_subtype'].isna()].obs.copy()
ips_cc.groupby(['line', 'ips_subtype']).size().unstack()

In [ ]:
tmp_results = []
for d in donors:
    donor_meta = ips_cc[ips_cc['line'] == d]
    cc_plot = donor_meta.melt(id_vars=['ips_subtype'], value_vars=['S.Score', 'G2M.Score'])
    cc_plot['donor'] = d
    tmp_results.append(cc_plot)
cc = pd.concat(tmp_results)
cc.shape

In [ ]:
rows = []
for (d, v), x in cc.groupby(['donor', 'variable']):
    a = x.loc[x['ips_subtype'] == 'IPS_main', 'value']
    b = x.loc[x['ips_subtype'] == 'IPS_alt', 'value']
    rows.append([d, v, len(a), len(b), a.median(), b.median(), mannwhitneyu(a, b).pvalue])
cc_stats = pd.DataFrame(rows, columns=['donor', 'score', 'n_LE', 'n_SE', 'median_LE', 'median_SE', 'p'])
cc_stats

In [ ]:
cc.to_csv(f'{outdir}/ips_subtype_rna_cell_cycle.csv', sep='\t')
cc_stats.to_csv(f'{outdir}/ips_subtype_rna_cell_cycle_stats.csv', sep='\t', index=False)

### ED10F: pluripotency markers

In [ ]:
qc = sc.read_h5ad(f'{rnadir}/h5ad/raw/mct.passqc_nuclei.h5ad')
sc.pp.normalize_total(qc)
sc.pp.log1p(qc)
qc.obs['ips_subtype'] = early_ct['ips_subtype'].reindex(qc.obs.index)
ips_qc = qc[~qc.obs['ips_subtype'].isna()]
ips_qc.shape

In [ ]:
genes = [g for g in ['EPCAM', 'NANOG', 'DNMT3B', 'DPPA4'] if g in qc.var.index]
markers = ips_qc.obs[['ips_subtype', 'donor']].copy()
for g in genes:
    markers[g] = ips_qc[:, g].X.toarray().ravel()
to_plot = markers.melt(id_vars=['ips_subtype'], value_vars=genes)
to_plot.groupby(['variable', 'ips_subtype'])['value'].agg(['mean', lambda x: (x > 0).mean()])

In [ ]:
rows = []
for g in genes:
    a = markers.loc[markers['ips_subtype'] == 'IPS_main', g]
    b = markers.loc[markers['ips_subtype'] == 'IPS_alt', g]
    rows.append([g, a.mean(), b.mean(), (a > 0).mean(), (b > 0).mean(), mannwhitneyu(a, b).pvalue])
marker_stats = pd.DataFrame(rows, columns=['gene', 'mean_LE', 'mean_SE', 'frac_expr_LE', 'frac_expr_SE', 'p'])
marker_stats

In [ ]:
to_plot.to_csv(f'{outdir}/ips_subtypes_pluri_markers.csv', sep='\t')
marker_stats.to_csv(f'{outdir}/ips_subtypes_pluri_markers_stats.csv', sep='\t', index=False)

### ED10G: RNA trajectory score

In [ ]:
traj_score = pd.read_csv(f'{rnadir}/csv/label_transfer/snmct_traj_score.csv', sep='\t', index_col=0)
traj_score['ips_subtype'] = early_ct['ips_subtype'].reindex(traj_score.index)
ips_traj = traj_score[~traj_score['ips_subtype'].isna()].copy()
ips_traj.head()

In [ ]:
ips_traj.shape

In [ ]:
rows = []
for d in donors:
    x = ips_traj[ips_traj['donor'] == d]
    a = x.loc[x['ips_subtype'] == 'IPS_main', 'traj_score']
    b = x.loc[x['ips_subtype'] == 'IPS_alt', 'traj_score']
    rows.append([d, len(a), len(b), a.median(), b.median(), mannwhitneyu(a, b).pvalue])
traj_stats = pd.DataFrame(rows, columns=['donor', 'n_LE', 'n_SE', 'median_LE', 'median_SE', 'p'])
traj_stats

In [ ]:
ips_traj.to_csv(f'{outdir}/ips_subtype_rna_traj_score.csv', sep='\t')
traj_stats.to_csv(f'{outdir}/ips_subtype_rna_traj_score_stats.csv', sep='\t', index=False)

### ED10I: DEGs between iPSC-SE and iPSC-LE (wilcoxon, as in 02a_call_degs_ips_main_alt)

In [ ]:
raw = sc.read_h5ad(f'{rnadir}/h5ad/raw/mct.h5ad')
raw.obs['ips_cluster'] = early_ct['ips_subtype'].reindex(raw.obs.index)
ips_raw = raw[~raw.obs['ips_cluster'].isna()].copy()
ips_raw.obs['ips_cluster'].value_counts()

In [ ]:
sc.pp.normalize_total(ips_raw)
sc.pp.log1p(ips_raw)
sc.tl.rank_genes_groups(ips_raw, 'ips_cluster', method='wilcoxon', key_added='wilcoxon')
main = sc.get.rank_genes_groups_df(ips_raw, group='IPS_main', key='wilcoxon')
alt = sc.get.rank_genes_groups_df(ips_raw, group='IPS_alt', key='wilcoxon')
alt.head()

In [ ]:
thresh = 1e-3
sig = alt[alt['pvals_adj'] < thresh]
len(sig), (sig['logfoldchanges'] > 0).sum(), (sig['logfoldchanges'] < 0).sum()

In [ ]:
alt[(alt['pvals_adj'] < thresh) & (alt['logfoldchanges'] > 0)].sort_values('pvals_adj').head(30)

In [ ]:
alt['names'].isin(['ATAD5', 'DNA2', 'BRCA1', 'GINS1']).sum(), alt[alt['names'].isin(['ATAD5', 'DNA2', 'BRCA1', 'GINS1'])]

In [ ]:
main.to_csv(f'{outdir}/degs_ips_main.csv', sep='\t')
alt.to_csv(f'{outdir}/degs_ips_alt.csv', sep='\t')

### compare with published DEGs (all nuclei)

In [ ]:
pub = pd.read_csv(f'{rnadir}/csv/degs/ips_subtypes/ips_alt.csv', sep='\t', index_col=0)
pub_sig = pub[pub['pvals_adj'] < thresh]
len(pub_sig), (pub_sig['logfoldchanges'] > 0).sum(), len(set(pub_sig['names']) & set(sig['names']))

In [ ]:
m = pub.set_index('names')['logfoldchanges'].to_frame('published').join(alt.set_index('names')['logfoldchanges'].to_frame('early_ipsc'), how='inner')
m.corr()

In [ ]:
!date